# 第 51 课｜装饰器、重试、缓存与资源边界

本 Notebook 对应「365 天量化金融」第 51 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/51_youtube-16Ypll6C4Is/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟数据接口，不发起网络请求
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

计时、重试和缓存会横跨许多接口函数；上下文管理器负责确保文件或事务在异常时释放。目标是减少重复控制代码，同时让重试次数、缓存命中和异常仍然可观测。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 1. 计时装饰器必须在异常时也记录

装饰器接收函数并返回包装后的函数。`functools.wraps` 保留原函数名称和文档；`finally` 保证无论成功或失败都写入耗时记录。

In [2]:
# 为任意函数增加计时证据，同时保留原函数元数据。
from functools import wraps
import time

timing_log = []

def timed(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        status = "ok"
        try:
            return function(*args, **kwargs)
        except Exception:
            status = "failed"
            raise
        finally:
            timing_log.append({
                "function": function.__name__,
                "status": status,
                "elapsed_ms": (time.perf_counter() - start) * 1000,
            })
    return wrapper

@timed
def compute_returns(prices):
    return prices.pct_change(fill_method=None)

small_prices = pd.Series([100.0, 101.0, 99.0])
display(compute_returns(small_prices))
display(pd.DataFrame(timing_log))
assert timing_log[-1]["function"] == "compute_returns"

0         NaN
1    0.010000
2   -0.019802
dtype: float64

,function,status,elapsed_ms
0,compute_returns,ok,0.13725


## 2. 重试只处理明确的暂时性异常

指数退避公式为 `wait_k = min(cap, base × 2^k)`。教学中把 `sleep` 注入为零等待函数，既能验证次数和间隔，又不拖慢 Notebook。下单、付款等非幂等操作不能盲目重试。

In [3]:
# 模拟前两次超时、第三次成功，并记录退避序列。
def retry(max_attempts=3, base=0.1, cap=1.0, retry_on=(TimeoutError,), sleep=time.sleep):
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            waits = []
            for attempt in range(max_attempts):
                try:
                    return function(*args, **kwargs), waits
                except retry_on:
                    if attempt == max_attempts - 1:
                        raise
                    wait = min(cap, base * 2 ** attempt)
                    waits.append(wait)
                    sleep(wait)
        return wrapper
    return decorate

state = {"calls": 0}
@retry(max_attempts=4, base=0.25, cap=1.0, sleep=lambda seconds: None)
def flaky_fetch(symbol):
    state["calls"] += 1
    if state["calls"] < 3:
        raise TimeoutError("temporary")
    return {"symbol": symbol, "rows": 756}

payload, waits = flaky_fetch("DEMO_A")
print("结果:", payload, "等待计划:", waits, "调用次数:", state["calls"])
assert waits == [0.25, 0.5]
assert state["calls"] == 3

结果: {'symbol': 'DEMO_A', 'rows': 756} 等待计划: [0.25, 0.5] 调用次数: 3


## 3. 缓存键要包含数据版本和复权口径

函数参数相同不表示数据语义相同。缓存键还应包含 `as_of`、复权方式和适配器版本；命中率为 `hits / (hits + misses)`。

In [4]:
# 构造版本化缓存，重复请求产生一次 miss 和一次 hit。
cache = {}
cache_stats = {"hits": 0, "misses": 0}

def cached_price(symbol, as_of, adjustment, adapter_version):
    key = (symbol, as_of, adjustment, adapter_version)
    if key in cache:
        cache_stats["hits"] += 1
        return cache[key]
    cache_stats["misses"] += 1
    value = {"symbol": symbol, "as_of": as_of, "price": 100.0, "adjustment": adjustment}
    cache[key] = value
    return value

first = cached_price("DEMO_A", "2025-01-02", "qfq", "v1")
second = cached_price("DEMO_A", "2025-01-02", "qfq", "v1")
hit_rate = cache_stats["hits"] / sum(cache_stats.values())
print("缓存统计:", cache_stats, "命中率:", hit_rate)
assert first is second
assert hit_rate == 0.5

缓存统计: {'hits': 1, 'misses': 1} 命中率: 0.5


## 4. 上下文管理器保证异常时释放资源

`contextlib.contextmanager` 中 `yield` 前是获取资源，`finally` 是释放资源。下面故意抛错，退出记录仍然出现。

In [5]:
# 捕获教学异常，并验证资源总会关闭。
from contextlib import contextmanager

resource_log = []
@contextmanager
def research_resource(name):
    resource_log.append(f"open:{name}")
    try:
        yield {"name": name}
    finally:
        resource_log.append(f"close:{name}")

try:
    with research_resource("snapshot") as resource:
        print("使用:", resource)
        raise ValueError("teaching failure")
except ValueError:
    print("已捕获教学异常")

print(resource_log)
assert resource_log == ["open:snapshot", "close:snapshot"]

使用: {'name': 'snapshot'}
已捕获教学异常
['open:snapshot', 'close:snapshot']


## 方法边界与复用结论


- 装饰器层数过多会隐藏控制流，必须输出计时、重试和缓存指标。
- 只有暂时性、幂等操作适合自动重试。
- 缓存必须有版本、失效条件、时区与公司行动更新策略。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。